# Advisor：让正在工作的智能体在回合中途咨询更强的模型

每个回合都调用你最强的模型，是安全的做法，也是最贵的做法。用中档模型很便宜，直到智能体撞上任务里真正需要判断力的那个决定，而那时它已经做出了承诺。常见的补救办法是自己搭一套升级机制：识别难题、打包上下文、调用更大的模型、把答案拼接回去。

Advisor 在会话内部完成这件事。你在智能体 `multiagent` 名单里用一个 `{"type": "advisor", "model": ...}` 条目指名第二个、能力更强的模型。于是主线程就拥有一个可以在回合中途调用的 `advisor` 工具：平台把到此为止的对话放到顾问模型面前，把它的建议返回给工作模型，采样继续。每次咨询都作为一个短生命周期的平台线程运行，有自己的词元用量，所以你能看到它何时发生，并准确核算它花了多少。

本笔记本讲解：

- 在智能体中加入 Advisor 条目
- 告诉工作模型何时应当咨询
- 从事件流读取一次咨询，并从线程读取它的成本
- 当 Advisor 的输出以脱敏形式返回时会发生什么

## 1. 配置客户端

Advisor 条目位于 `multiagent` 块中，使用标准的 `managed-agents-2026-04-01` 测试版请求头。带类型的名单结构需要 `anthropic>=0.121.0`。

In [1]:
%%capture
%pip install -qU "anthropic>=0.121.0" python-dotenv

In [2]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
WORKER_MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-5")
ADVISOR_MODEL = os.environ.get("COOKBOOK_ADVISOR_MODEL", "claude-opus-5")
client = anthropic.Anthropic()

## 2. 给 Sonnet 智能体配一个 Opus 顾问

你在名单里声明 Advisor，而不是把它作为工具定义，也不是作为单独创建的智能体。该条目与任何专家一起放在 `multiagent.agents` 里；只含 Advisor 条目的名单也是合法的：智能体会咨询，但不会派生子智能体。服务器在保存时强制以下几条规则：

- 每个名单最多一个 Advisor 条目。
- `model` 必须是允许充当 Advisor 的模型，且智能体自身模型与它的搭配必须是平台允许的搭配。不合规的搭配会在 `agents.create` 上返回 400，而不是运行时的意外。
- 该条目在名单中占用名字 `anthropic.advisor`，因此任何专家都不能使用这个名字。

运行时，平台把该条目以 `advisor` 工具的形式呈现给工作模型。这个工具不接受输入，因为顾问读取的是到调用为止的整段对话，而不是工作模型写出的查询。因此咨询策略要写进系统提示词：它控制的是模型何时去用这个工具。

In [3]:
SYSTEM = """You are a backend engineer designing HTTP APIs.
You have an advisor: a more capable model that can review your conversation so far and
send back guidance. Consult it with the advisor tool before you commit to any decision
that would be expensive to reverse once clients depend on it: identifier and idempotency
schemes, pagination contracts, error semantics, versioning. Do routine drafting yourself.
When you consult, act on the guidance you get back and say what you changed.
Write your final design to /mnt/session/outputs/design.md."""

designer = client.beta.agents.create(
    name="api_designer",
    description="Designs HTTP APIs, escalating irreversible decisions to an advisor.",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-consult-an-advisor"},
    model={"id": WORKER_MODEL},
    system=SYSTEM,
    tools=[{"type": "agent_toolset_20260401"}],
    multiagent={
        "type": "coordinator",
        "agents": [
            {"type": "advisor", "model": ADVISOR_MODEL},
        ],
    },
    betas=BETAS,
)
print(f"{designer.name}: {designer.id} v{designer.version}")
print("roster:", [entry.to_dict() for entry in designer.multiagent.agents])

api_designer: agent_staging_018FvRjvrM6ASGuXXcDTAvdH v1
roster: [{'model': 'claude-opus-5', 'type': 'advisor'}]


## 3. 交给它一个内含艰难抉择的任务

为一个涉及资金的端点设计幂等方案，正是提示词要求智能体上报的那类决定：写起来便宜，客户据此发布之后改动就很贵。设计的其余部分属于常规内容，所以校准良好的智能体会就这一点咨询，其余部分自己起草。

咨询开销只受发起咨询的那个回合限制：Advisor 的词元在工作模型的词元之外另行计费，而且没有单次咨询上限。会话带有 `budget`，因此过于积极的升级习惯会在已知上限处暂停，而不是把账单越滚越大（机制见 [`CMA_cap_session_spend.ipynb`](CMA_cap_session_spend.ipynb)）。

In [4]:
env = client.beta.environments.create(
    name="advisor-demo",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)

session = client.beta.sessions.create(
    agent=designer.id,
    environment_id=env.id,
    title="Design: refunds API",
    budget={"type": "limit", "max_list_cost": {"currency": "USD", "amount": "500"}},
    initial_events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": "Design the REST surface for a refunds resource on a payments "
                    "API: create, retrieve, list. Refund creation is retried by mobile "
                    "clients on flaky networks, and a double refund costs real money. "
                    "Cover the request/response shapes, idempotency, pagination on list, "
                    "and error codes.",
                }
            ],
        }
    ],
    betas=BETAS,
)
print(session.id, session.status)

sesn_staging_013TqQwzrbUZDX1KJW2b9KnC running


## 4. 在主事件流上观察一次咨询

咨询不是 `agent.tool_use` 事件。它以主事件流上的线程生命周期形式出现，`agent_name` 设为 `anthropic.advisor`：

```
session.thread_created            agent_name: anthropic.advisor
session.thread_status_running     agent_name: anthropic.advisor
agent.thread_message_received     from_agent_name: anthropic.advisor, content: [{type: "text", ...}]
session.thread_status_idle        agent_name: anthropic.advisor
session.thread_status_terminated  agent_name: anthropic.advisor
```

咨询线程自始至终处于 `running` 状态。建议在调用点交付，采样随即继续，所以从外部看：主线程暂停，一个 Advisor 线程出现又消失，主线程带着已并入的建议恢复。每次咨询都是一个新线程，交付后自行终止。

In [5]:
def advice_text(content):
    """Render an advisor delivery: joined text, or a marker for the redacted arm."""
    if any(block.type == "redacted" for block in content):
        return "[redacted advice: content withheld by the advisor model's policy]"
    return "".join(block.text for block in content if block.type == "text")


consultations = 0

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "session.thread_created" and ev.agent_name == "anthropic.advisor":
            consultations += 1
            print(f"\n--- consulting advisor #{consultations} ({ev.session_thread_id}) ---")
        elif (
            ev.type == "agent.thread_message_received" and ev.from_agent_name == "anthropic.advisor"
        ):
            print(advice_text(ev.content)[:800])
            print("--- advice received, primary thread resumes ---\n")
        elif ev.type == "agent.message":
            print("".join(b.text for b in ev.content if b.type == "text"), end="")
        elif ev.type == "session.status_idle":
            print(f"\n[idle] stop_reason={ev.stop_reason.type}")
            break

print(f"\nconsultations this turn: {consultations}")


--- consulting advisor #1 (sthr_staging_01PnQEQPc1nLxUK4S2vcVbCL) ---


[redacted advice: content withheld by the advisor model's policy]
--- advice received, primary thread resumes ---



Incorporated the advisor's feedback directly into the design — the most consequential changes were:

- **Idempotency became a stored resource with a claim→execute→finalize lifecycle**, not a simple header check. This closes the "processor succeeded but our server died before recording it" window, which a naive check-then-execute design leaves open.
- **Added a second, independent safety layer**: a server-side invariant (`sum(active refunds) + new_amount <= captured_amount`) enforced under a row lock, so the system is safe even if a client reuses/loses its idempotency key.
- **Made refund creation explicitly asynchronous** (`201` + `status: pending`, terminal states `succeeded|failed|canceled`) instead of implying synchronous success.
- **Added a recovery path** (`GET /refunds?idempotency_key=...`) for clients that lose the response to a request that actually succeeded — critical for flaky mobile networks.
- **Switched list pagination to opaque cursor-based** instead of offset, since re

The design is written to `/mnt/session/outputs/design.md`.

**Summary of the approach:**

- **Endpoints:** `POST /v1/payments/{payment_id}/refunds` (create, scoped to parent), `GET /v1/refunds/{id}` (retrieve), `GET /v1/refunds` (list) — top-level for retrieve/list per advisor guidance.
- **Idempotency — the centerpiece:** `Idempotency-Key` header is required and persisted as its own resource with a **claim → execute → finalize** lifecycle (claim written in the same transaction as the pending refund), so a crash after the processor succeeds but before we record it can't produce a silent duplicate on retry. Replay rules are explicit: same key+body replays the original response, same key+different body is a `422`, same key still in-flight is a `409`.
- **Second safety net, independent of the header:** a server-enforced invariant (`sum(active refunds) + new_amount <= captured_amount`) under a row lock, protecting against buggy clients that regenerate keys or omit them.
- **Async-first sta

## 5. 为咨询计价

因为每次咨询都是真实的会话线程，它的成本挂在该线程上，而不是隐形地并入主线程的数字。用 `session.thread_created` 事件里的 id 逐个取出 Advisor 线程，读取它的 `usage`。词元数在那里，`usage.list_cost` 也在那里：该线程的词元按 Advisor 模型的公开牌价计价，以 USD 最小货币单位的整数字符串表示（`"131"` 即 $1.31），因此成本来自 API，而不是你自己维护的价目表。这些词元在主线程之外另行计费，并且已经计入会话级的 `usage` 汇总，后者自带 `list_cost`。

一次完成的咨询就是一个 `terminated` 线程。它仍可取回，但它的 `agent` 不是通常那种快照后的智能体：平台 Advisor 背后没有 Agent 资源，所以线程携带的是名单条目本身，`{"type": "advisor", "model": ...}`。这个 `agent.type` 取值 `"advisor"` 是区分平台 Advisor 线程与普通子线程的可靠方式，比名字更可靠。

In [6]:
from decimal import Decimal


def usd(money) -> str:
    """Render an integer minor-unit amount ("131" = $1.31) as dollars."""
    return f"${Decimal(money.amount) / 100:.2f}"


advisor_thread_ids = [
    ev.session_thread_id
    for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS)
    if ev.type == "session.thread_created" and ev.agent_name == "anthropic.advisor"
]

advisor_in = advisor_out = 0
advisor_cents = Decimal("0")
for thread_id in advisor_thread_ids:
    thread = client.beta.sessions.threads.retrieve(thread_id, session_id=session.id, betas=BETAS)
    usage = thread.usage
    advisor_in += usage.input_tokens or 0
    advisor_out += usage.output_tokens or 0
    advisor_cents += Decimal(usage.list_cost.amount)
    print(
        f"{thread.id}  agent.type={thread.agent.type}  model={thread.agent.model}  "
        f"status={thread.status}  "
        f"in={usage.input_tokens} out={usage.output_tokens}  {usd(usage.list_cost)}"
    )

total = client.beta.sessions.retrieve(session.id, betas=BETAS).usage
print(f"\nadvisor tokens: in={advisor_in} out={advisor_out}  cost=${advisor_cents / 100:.2f}")
print(
    f"session total:  in={total.input_tokens} out={total.output_tokens}  "
    f"cost={usd(total.list_cost)}  (advisor included)"
)

sthr_staging_01PnQEQPc1nLxUK4S2vcVbCL  agent.type=advisor  model=claude-opus-5  status=terminated  in=603 out=2468  $0.07

advisor tokens: in=603 out=2468  cost=$0.07
session total:  in=15553 out=8491  cost=$0.21  (advisor included)


## 6. 阅读设计文档

智能体用 `write` 工具写出了最终文档。从事件日志里取出它，看幂等性那一节，咨询正是落在那里。

In [7]:
design = ""
for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if (
        ev.type == "agent.tool_use"
        and ev.name == "write"
        and ev.input.get("file_path", "").endswith("design.md")
    ):
        design = ev.input["content"]  # keep the last write, in case of revisions

for line in design.splitlines():
    if line.startswith("#"):
        print(line)
print()
print(design[:1200])

# Refunds API Design
## 1. Resources & endpoints
## 2. Idempotency (the core problem: safe retries on flaky mobile networks)
### 2.1 Header contract
### 2.2 Server-side lifecycle (claim → execute → finalize)
### 2.3 Replay semantics (what a retry actually gets back)
### 2.4 Second, independent safety layer: a business invariant
### 2.5 Recovery path for lost responses
## 3. Request / response shapes
### 3.1 Create
### 3.2 Retrieve
### 3.3 List
## 4. Error model

# Refunds API Design

## 1. Resources & endpoints

| Action | Method & path |
|---|---|
| Create a refund | `POST /v1/payments/{payment_id}/refunds` |
| Retrieve a refund | `GET /v1/refunds/{refund_id}` |
| List refunds | `GET /v1/refunds` |
| Look up by idempotency key (recovery) | `GET /v1/refunds?idempotency_key={key}` |

**Why creation is nested but retrieve/list are top-level:** creation under the parent
gives a natural scope for the amount-conservation invariant ("refunds of this payment")
and a clean `404` when `payment_

## 7. 当建议以脱敏形式返回

你能否读到一次咨询，是 Advisor 模型的属性，由与 Messages API 的 advisor 工具相同的 Anthropic 策略决定。对于策略会隐藏输出的模型，交付事件携带的是 `[{"type": "redacted"}]` 占位块而不是文本，Advisor 线程自身的 `agent.message` 事件也是占位内容。

脱敏改变的是你能观察到什么，不是智能体实际使用了什么。无论哪种策略，工作模型都会完整读取建议，所以设计仍然得到改进。所有结构性内容依然可见：线程、它的状态流转、它的计时和它的 `usage`，这也是第 4 步里的 `advice_text` 辅助函数渲染一个标记而不是崩溃的原因。Advisor 自身的推理在两种策略下都不会交付，只交付它的最终文本。

既然策略可以在不改 API 的情况下变化，那么在一切消费这些事件的代码里，就应把 `text` 和 `redacted` 两个分支都视为完全正常。

## 8. 与专家团队并存的 Advisor

Advisor 条目与普通名单成员共享 `agents` 数组，因此协调者可以在同一个会话里既向专家分派任务，又咨询更强的模型。只有主线程会咨询。它派生的专家自身不带 Advisor 运行，且 Advisor 线程不受子线程并发上限约束，所以已经达到子线程上限的协调者仍然可以咨询。

```python
client.beta.agents.create(
    name="research_coordinator",
    model={"id": WORKER_MODEL},
    system="Delegate research to your specialists, then synthesize their reports.",
    tools=[{"type": "agent_toolset_20260401"}],
    multiagent={
        "type": "coordinator",
        "agents": [
            "agent_01Res...",                                # a specialist, by id
            {"type": "agent", "id": "agent_01Aud...", "version": 3},
            {"type": "advisor", "model": ADVISOR_MODEL},     # the advisor
        ],
    },
    betas=BETAS,
)
```

协调者模式本身、按角色的工具作用域划分以及分派事件类型，在 [`CMA_coordinate_specialist_team.ipynb`](CMA_coordinate_specialist_team.ipynb) 中已有讲解。Advisor 在会话创建时即已固定：改动智能体上的 Advisor 只影响之后创建的会话，不影响已在运行的那个会话。

## 9. 清理

In [8]:
from utilities import wait_for_idle_status

wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id, betas=BETAS)
client.beta.environments.archive(env.id, betas=BETAS)
print("archived")

archived
